# dev readChannelDescriptions

Mark Dransfield 9 April 2024

This notebook is a test bed for the development of the `readChannelDescriptions` code which will read and decode the ChannelDescriptions.txt files delivered by Xcalibur, and save the information into the desired `whizzFile` as per `updateChannels`. This latter will be modified to call `readChannelDescriptions` as required. 

Summary cells are coloured to quickly indicated status against QC. The colours are:

<div class="alert alert-block alert-success">
Delivery passed QC.
<div>

<div class="alert alert-block alert-info">
Delivery was not checked, or the check is not applicable, or there is a minor shortcoming but is the data are acceptable.
<div>

<div class="alert alert-block alert-warning">
Delivery failed QC and problem has been reported for rectification.
<div>

<div class="alert alert-block alert-danger">
Delivery failed QC and problem might not be readily rectified. This is very rare.
<div>

#### Import required modules, and set filenames for data and plan.

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib widget
from pathlib import Path

In [2]:
import AirGravQC as qc

In [96]:
data_root = "/Users/markdransfield/Documents/GitHub/AirGravQC/examples/FalconAGG/"
dx = Path(data_root + r'20211130.xyz')
dh = dx.with_suffix('.hdf5')
cd = Path(data_root + r'Channel Descriptions.txt')

sglxyz = Path(r'/Users/markdransfield/Documents/GitHub/AirGravQC/examples/AirGrav/FD003_SGL/DLV003_GA5371.xyz')
readme = Path(r'/Users/markdransfield/Documents/GitHub/AirGravQC/examples/AirGrav/FD003_SGL/ReadMe_Grav.txt')
sglwhizz = Path(r'/Users/markdransfield/Documents/GitHub/AirGravQC/examples/AirGrav/FD003_SGL/DLV003_GA5371.hdf5')


#### For SGL data

In [112]:
def updateWithReadMe(readme, whizzfile, verbose=False):
    """
    Get the channel name, units and description from the `ReadMe_Grav.txt` file and
    update those channels in the `whizzfile`.
    """
    with open(readme, 'r') as fid:
        columnheaderfound = False
        channeldetailsfound = False
        for file_line in fid:
            # print(file_line)
            if 'Name' in file_line and 'Units' in file_line and 'Description' in file_line:
                columnheaderfound = True
                namecolumn = 1
                unitscolumn = 2
                desccolumn = 4
                continue
            if not columnheaderfound:
                continue
            if len(file_line.strip()) < 2:
                if channeldetailsfound:
                    break
                else:
                    continue
            data_rec = file_line.split()
            # print(data_rec)
            if len(data_rec) > 4:
                channeldetailsfound = True
                name = trim(data_rec[namecolumn])
                name = name.replace('-','_') # to match xyzToHDF
                file_line = file_line[file_line.index(data_rec[namecolumn]) :]
                units = trim(data_rec[unitscolumn])
                if units == '-':
                    units = ''
                file_line = file_line[file_line.index(data_rec[unitscolumn]) :]
                description = trim(file_line[file_line.index(data_rec[desccolumn]) :])
                try:
                    qc.updateChannelAttributes(whizzfile, name, units=units, description=description)
                    if verbose:
                        print (f'  NAME: {name}\n  UNITS: {units}\n  DESCRIPTION: {description}\n')
                except:
                    print(f'  Channel {name} could not be updated.')
                    continue

In [113]:
updateWithReadMe(readme, sglwhizz, verbose=True)

Changed channel attribute(s) for LINE in DLV003_GA5371.hdf5.
  NAME: LINE
  UNITS: 
  DESCRIPTION: Line Number XXXX.YY where XXXX is line number and YY is segment or reflight number

Changed channel attribute(s) for FLIGHT in DLV003_GA5371.hdf5.
  NAME: FLIGHT
  UNITS: 
  DESCRIPTION: Flight Number

Changed channel attribute(s) for YEAR in DLV003_GA5371.hdf5.
  NAME: YEAR
  UNITS: 
  DESCRIPTION: Year

Changed channel attribute(s) for DOY in DLV003_GA5371.hdf5.
  NAME: DOY
  UNITS: 
  DESCRIPTION: Day of year

Changed channel attribute(s) for FTIME in DLV003_GA5371.hdf5.
  NAME: FTIME
  UNITS: s
  DESCRIPTION: Fiducial Seconds Past Midnight UTC

Changed channel attribute(s) for MGA_X in DLV003_GA5371.hdf5.
  NAME: MGA_X
  UNITS: m
  DESCRIPTION: X coordinate, GDA2020 / MGA zone 55

Changed channel attribute(s) for MGA_Y in DLV003_GA5371.hdf5.
  NAME: MGA_Y
  UNITS: m
  DESCRIPTION: Y coordinate, GDA2020 / MGA zone 55

Changed channel attribute(s) for MGA_Z in DLV003_GA5371.hdf5.
  NAME

In [118]:
block_name = 'Test SGL Data'
qc.updateProject(sglwhizz, acquirer='SGL', blockID=block_name)
qc.updateCoordFrame(sglwhizz, lat='LAT', lon='LONG', x='MGA_X', y='MGA_Y', time='FTIME', alt='MGA_Z')
qc.updateCoordFrame(sglwhizz, geoDatum='WGS84', htDatum='GRS80', projection='UTM', utmz='54')

Setting BlockID = Test SGL Data for DLV003_GA5371.hdf5.
Setting Acquirer = SGL for DLV003_GA5371.hdf5.
Changed CoordFrame attribute(s) for DLV003_GA5371.hdf5.
Changed CoordFrame attribute(s) for DLV003_GA5371.hdf5.


In [119]:
qc.reportWhizz(sglwhizz, channel='FTIME')

Whizz Version 1.0
    Acquirer: SGL
    BlockID: Test SGL Data
    ProjectName: 5371

Coordinates
    AltitudeChannel: MGA_Z
    GeoDatum: WGS84
    HeightDatum: GRS80
    LatitudeChannel: LAT
    LongitudeChannel: LONG
    Projection: UTM
    TimeChannel: FTIME
    UTMZone: 54
    XChannel: MGA_X
    YChannel: MGA_Y
14 lines: total distance flown [km] = 1,405.9

14 lines:
 ['2259.000', '2261.000', '2263.000', '2265.000', '2267.000', '2269.000', '2271.000', '2273.000', '2275.000', '2277.000', '2279.000', '2281.000', '2283.000', '2303.000']

34 channels:
 ['ATMCOR', 'B100s_267_GEOID', 'B100s_267_GRS80', 'B56s_267_GEOID', 'B56s_267_GRS80', 'DEM', 'DOY', 'EOTCOR', 'FA100s_GEOID', 'FA100s_GRS80', 'FA56s_GEOID', 'FA56s_GRS80', 'FACOR_GEOID', 'FACOR_GRS80', 'FLIGHT', 'FTIME', 'FX', 'FY', 'FZ', 'LALT', 'LAT', 'LATCOR', 'LINE', 'LONG', 'MGA_X', 'MGA_Y', 'MGA_Z', 'MSL_Z', 'RALT', 'STATCOR', 'TACOR', 'V_EAST', 'V_NORTH', 'YEAR']

Channel <HDF5 dataset "FTIME": shape (2732,), type "<f8">
    Desc

#### For Xcalibur data

In [114]:
def trim(mystring):
    """
    Remove leading spaces, trailing \n, and then trailing spaces from `mystring`.
    """
    return mystring.lstrip(' ').rstrip(' ').rstrip('\n')

In [115]:
def updateWithChannelDescriptions(cd_textfile, whizzfile):
    """
    Get the channel name, units and description from the `ChannelDescriptions.txt` file and
    update those channels in the `whizzfile`.
    """
    with open(cd_textfile, 'r') as fid:
        for file_line in fid:
            data_rec = file_line.split('|')
            if len(data_rec) == 3:
                name = trim(data_rec[0])
                units = trim(data_rec[1])
                description = trim(data_rec[2])
                # print(f'name = {name}, units = {units}, description = {description}')
                if name != 'ChannelName':
                    qc.updateChannelAttributes(whizzfile, name, units=units, description=description)


In [116]:
updateWithChannelDescriptions(cd, dh)

KeyError: "Unable to synchronously open object (object 'Altitude_Geoid09' doesn't exist)"

In [ ]:
qc.reportWhizz(dh)

#### Point located data format
<div class="alert alert-block alert-info">
ACCEPT
<div>

*Delivered point located field data shall be in ASEG-GDF2 format.*

Actually delivered in Geosoft XYZ. For field data this is not a problem so, even though this does not meet contractual requirements, it is acceptable. Channel descriptions file matches actual supplied channels as required.

#### Gridded data format
<div class="alert alert-block alert-info">
NA   
<div>

*The Deed specifies the gridded field data to be supplied. For the Canobie contract, none were required.*

> The Deed specifies ERMapper format.

No grids required and none supplied. Not applicable.

#### Static data to be supplied
<div class="alert alert-block alert-success">
PASS
<div>

*Gravity and navigation processed static readings must be reported to the Customer within 24 hours of every completed flight.*

Quiescent data has been supplied as required. This has been QC'd on a daily basis and no problems, concerns or failings were observed. All data has passed.

#### Field data to be supplied
<div class="alert alert-block alert-success">
PASS
<div>

*The Deed specifies the required channels for field delivery.*

Line data - all required channels delivered.

Channel description text file supplied.

#### Delivered reports
<div class="alert alert-block alert-success">
PASS
<div>

*The Deed requires a Commencement Report before acquisition commences, daily and weekly report through the project, and a final report at the end.*

Commencement report was received in timely manner and was checked against contract. All requirements met.

Daily reports have been received and have provided the information required under the contract.

Weekly reports - none received, supplier has been asked to rectify (email 30 Feb 2019).

Final report - only due at end of project.

#### Coordinates and Units
<div class="alert alert-block alert-info">
ACCEPT
<div>

*As per contract: Correct Coordinate Frame; correct Position Datum, Projected Coordinates and Height Datum; correct units.*

Checked manually.

Coord Frame - PASS

Position Datum - WGS84 incorrect but acceptable for field data.

Projected Coordinate - UTM 54 incorrect (should be MGA 54) but acceptable for field data.

Height Datum - EGM96 geoid incorrect but acceptable for field data.

Units - all correct except gravity which is in mGal instead of µm/s/s. Accept since field data.

> I accept WGS84/UTM and I accept mGal for field data even though the Deed specifies GDA2020 / MGA and um/s/s. Enforcing the contract makes extra work for the acquirer on every delivery for usually no advantage.